# Experiment 2.1 — SNN training-objective ablation

## Research question

With the same no-bias feed-forward SNN, same unsigned event input, same neuron dynamics, same split, same optimizer, and paired initialization:

\[
\boxed{\text{Which training objective can reliably make the SNN learn?}}
\]

This comes **before** readout / width / depth sweeps because chance-level training performance makes later capacity comparisons uninterpretable.

### Fixed backbone

\[
C \rightarrow 64 \rightarrow 64 \rightarrow K
\]

Fixed across every objective:

- unsigned polarity-split weighted events;
- dynamic event channel count \(C\);
- no recurrence;
- **SNN Linear bias = False**;
- single fixed \(\tau_{syn}\);
- single fixed \(\tau_{mem}\);
- fixed threshold;
- fixed surrogate gradient;
- one fixed user-disjoint split;
- same paired seeds.

Zero-input firing should remain 0.

---

## Objective A — `timestep_ce`

Existing repo-style control:

\[
L_A =
\frac{1}{N_{valid}}
\sum_{t\in valid}CE(s_t,y)
\]

This asks every valid timestep to support the final letter label.

## Objective B — `whole_count_ce`

\[
c_k = \sum_{t\in valid}s_{t,k}
\]

\[
L_B = CE(c,y)
\]

This is the simplest whole-gesture classification objective.

## Objective C — `relative_10bin_ce`

Divide every gesture into ten equal relative-progress bins:

\[
L_C =
CE(W\,vec(Z_{rel10})+b,\ y)
\]

This preserves gesture phase but is not naturally streaming because the final duration is required.

## Objective D — `hybrid_relative10bin_count`

\[
L_D =
L_C + \lambda L_B
\]

Default \(\lambda=0.1\).

## Objective E — `fixed_200ms_ce`

Use causal fixed-duration bins:

```text
0–200 ms
200–400 ms
400–600 ms
...
```

At 200 Hz this is 40 samples/bin.

\[
L_E =
CE(W\,vec(Z_{200ms})+b,\ y)
\]

This is the primary real-time-friendly temporal objective.

## Objective F — `last_200ms_ce`

Only the final valid 200 ms contributes:

\[
L_F =
CE(c_{last200ms}, y)
\]

This tests whether the gesture ending alone is sufficient. With the current FF SNN lacking true long-term recurrent memory, a poor F but strong E would support the hypothesis that multiple local patterns must be integrated across the gesture.

---

## Important training rule

**No early stopping.**

The repo-style objective can remain silent for many epochs before firing begins. Every objective therefore gets the same full epoch budget.

After training all epochs, the checkpoint with the **minimum validation loss** is restored.

This avoids prematurely killing a model at epoch 9 simply because its firing has not started yet.


In [ ]:
from __future__ import annotations

from pathlib import Path
import copy, hashlib, json, math, os, random, sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from IPython.display import display

import snntorch as snn
from snntorch import surrogate
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "snn").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the writingRing repository root")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from snn.accel_reconstruction_eval.datasets import load_acceleration_data

print("Repository root:", REPO_ROOT)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


## 1. Configuration

In [ ]:
DATASET_ROOTS = [
    REPO_ROOT / "outputs/action0_wavelets_1_2_4_8_16",
    REPO_ROOT / "outputs/action1_wavelets_1_2_4_8_16",
]

INCLUDED_LABELS = (
    "A", "B", "C", "D", "E", "X",
    "G", "H", "I", "J", "K", "L",
)

SPLIT_SEED = 12345
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15

SEEDS = (11, 23, 101, 40, 231)

OBJECTIVES_TO_RUN = (
    "timestep_ce",
    "whole_count_ce",
    "relative_10bin_ce",
    "hybrid_relative10bin_count",
    "fixed_200ms_ce",
    "last_200ms_ce",
)

BATCH_SIZE = 64
NUM_WORKERS = 0

# Same compute budget for every objective. No early stopping.
NUM_EPOCHS = 150
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.0
GRAD_CLIP_NORM = None

FIRING_ONSET_THRESHOLD = 1e-3

HIDDEN_WIDTH = 64
N_RELATIVE_BINS = 10
FIXED_BIN_MS = 200.0
HYBRID_COUNT_WEIGHT = 0.1

TAU_SYN_MS = 77.47
TAU_MEM_MS = 7.213
THRESHOLD = 1.0
SURROGATE_SLOPE = 25.0
RESET_MECHANISM = "subtract"

SPIKE_REGULARIZATION = 0.0
ZERO_INPUT_DIAGNOSTIC_STEPS = 200

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EXPERIMENT_ID = "experiment_2_1_snn_training_objective_ablation"
RESULTS_DIR = (
    REPO_ROOT
    / "notebooks/artifacts/experiment_2_1_snn_training_objective_ablation"
)

print("Objectives:", OBJECTIVES_TO_RUN)
print("Seeds:", SEEDS)
print("Full SNN trainings:", len(OBJECTIVES_TO_RUN) * len(SEEDS))
print("Device:", DEVICE)


## 2. Reproducibility

In [ ]:
def derive_seed(master_seed: int, *parts: object) -> int:
    text = "|".join([str(master_seed), *(str(p) for p in parts)])
    digest = hashlib.sha256(text.encode("utf-8")).digest()
    return int.from_bytes(digest[:4], "little", signed=False)


def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError:
        torch.use_deterministic_algorithms(True)


def worker_init_fn(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


## 3. Load and validate unsigned dynamic-channel event data

In [ ]:
data = load_acceleration_data(
    DATASET_ROOTS,
    repository_root=REPO_ROOT,
    require_reconstruction=False,
)

sampling_rates = {float(m.sampling_rate_hz) for m in data.producer_metadatas}
if len(sampling_rates) != 1:
    raise ValueError(f"Expected one shared sampling rate, got {sampling_rates}")
SAMPLING_RATE_HZ = sampling_rates.pop()

event_contracts = set()
for metadata in data.producer_metadatas:
    raw = metadata.raw
    event_representation = raw.get("event_representation")
    event_feature_schema = raw.get("event_feature_schema")
    event_channel_count = raw.get("event_channel_count")
    encoder_spec_sha256 = raw.get("spike_encoder_spec_sha256")

    if event_representation != "unsigned":
        raise ValueError(
            "Requires event_representation='unsigned'; "
            f"got {event_representation!r}"
        )
    if not isinstance(event_feature_schema, str) or not event_feature_schema:
        raise ValueError("Missing event_feature_schema")
    if not isinstance(event_channel_count, int) or event_channel_count <= 0:
        raise ValueError("event_channel_count must be positive")
    if event_channel_count != metadata.channel_count - 6:
        raise ValueError(
            "Expected six auxiliary IMU channels after event channels; "
            f"event={event_channel_count}, total={metadata.channel_count}"
        )
    if not isinstance(encoder_spec_sha256, str) or len(encoder_spec_sha256) != 64:
        raise ValueError("Missing spike_encoder_spec_sha256")

    event_contracts.add(
        (
            event_representation,
            event_feature_schema,
            event_channel_count,
            encoder_spec_sha256,
        )
    )

if len(event_contracts) != 1:
    raise ValueError(f"Incompatible event contracts: {event_contracts}")

(
    EVENT_REPRESENTATION,
    EVENT_FEATURE_SCHEMA,
    INPUT_CHANNELS,
    ENCODER_SPEC_SHA256,
) = event_contracts.pop()

rows = []
included = None if INCLUDED_LABELS is None else set(map(str, INCLUDED_LABELS))
padded_lengths = set()

for package_index, package in enumerate(data.packages):
    padded_lengths.add(int(package.padded_spike_imu.shape[1]))

    for segment_index, label in enumerate(package.labels.astype(str)):
        label = str(label)
        if included is not None and label not in included:
            continue

        valid_length = int(package.valid_lengths[segment_index])
        if valid_length <= 0:
            raise ValueError("valid_length must be positive")

        event_values = np.asarray(
            package.padded_spike_imu[
                segment_index, :valid_length, :INPUT_CHANNELS
            ]
        )
        if np.any(~np.isfinite(event_values)):
            raise ValueError("Non-finite event values")
        if np.any(event_values < 0.0):
            raise ValueError(
                f"Unsigned event contract violated for "
                f"{package.user}/action_{package.action}/{segment_index}"
            )

        rows.append(
            {
                "package_index": package_index,
                "segment_index": segment_index,
                "user": str(package.user),
                "action": str(package.action),
                "label": label,
                "valid_length": valid_length,
                "sample_id": f"{package.user}/action_{package.action}/{segment_index}",
            }
        )

if len(padded_lengths) != 1:
    raise ValueError(f"Expected one padded length, got {padded_lengths}")
PADDED_LENGTH = padded_lengths.pop()

manifest = pd.DataFrame(rows)
if manifest.empty:
    raise ValueError("No samples remain after label filtering")

labels_sorted = sorted(manifest["label"].unique().tolist())
CLASS_TO_IDX = {label: i for i, label in enumerate(labels_sorted)}
manifest["label_idx"] = manifest["label"].map(CLASS_TO_IDX).astype(int)
NUM_CLASSES = len(CLASS_TO_IDX)

DT_MS = 1000.0 / SAMPLING_RATE_HZ
ALPHA = float(math.exp(-DT_MS / TAU_SYN_MS))
BETA = float(math.exp(-DT_MS / TAU_MEM_MS))

FIXED_BIN_STEPS = max(
    1,
    int(round(FIXED_BIN_MS * SAMPLING_RATE_HZ / 1000.0)),
)
FIXED_BIN_COUNT = int(math.ceil(PADDED_LENGTH / FIXED_BIN_STEPS))

print(f"Samples: {len(manifest):,}")
print("Users:", manifest.user.nunique())
print("Classes:", NUM_CLASSES, labels_sorted)
print("Input event channels:", INPUT_CHANNELS)
print("Sampling rate:", SAMPLING_RATE_HZ, "Hz")
print("Padded length:", PADDED_LENGTH)
print(
    f"Fixed bin: {FIXED_BIN_MS:.1f} ms = "
    f"{FIXED_BIN_STEPS} samples, {FIXED_BIN_COUNT} bins"
)
print(f"alpha={ALPHA:.6f}, beta={BETA:.6f}, threshold={THRESHOLD}")


## 4. One fixed user-disjoint split

In [ ]:
def make_user_split(manifest: pd.DataFrame) -> pd.DataFrame:
    users = sorted(manifest["user"].unique().tolist())
    rng = np.random.default_rng(SPLIT_SEED)
    perm = np.array(users, dtype=object)
    rng.shuffle(perm)

    n = len(perm)
    n_train = max(1, int(np.floor(TRAIN_FRACTION * n)))
    n_val = max(1, int(np.floor(VAL_FRACTION * n)))
    if n_train + n_val >= n:
        n_train, n_val = n - 2, 1

    train_users = set(perm[:n_train].tolist())
    val_users = set(perm[n_train:n_train + n_val].tolist())

    out = manifest.copy()
    out["split"] = out["user"].map(
        lambda u: "train" if u in train_users else (
            "val" if u in val_users else "test"
        )
    )
    return out


FIXED_SPLIT_MANIFEST = make_user_split(manifest)

display(
    FIXED_SPLIT_MANIFEST.groupby("split").agg(
        users=("user", "nunique"),
        samples=("label", "size"),
        classes_present=("label", "nunique"),
    )
)


## 5. Dataset and paired DataLoaders

In [ ]:
class EventSNNDataset(Dataset):
    def __init__(self, data, subset_manifest: pd.DataFrame):
        self.data = data
        self.df = subset_manifest.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        package = self.data.packages[int(row.package_index)]
        segment_index = int(row.segment_index)
        valid_length = int(row.valid_length)

        x = np.asarray(
            package.padded_spike_imu[
                segment_index, :, :INPUT_CHANNELS
            ],
            dtype=np.float32,
        ).copy()

        if x.shape != (PADDED_LENGTH, INPUT_CHANNELS):
            raise ValueError(f"Unexpected input shape {x.shape}")
        if np.any(x[:valid_length] < 0.0):
            raise ValueError("Unsigned event contract violated")

        x[valid_length:] = 0.0
        valid_mask = np.arange(PADDED_LENGTH) < valid_length

        return {
            "x": torch.from_numpy(x),
            "label": torch.tensor(int(row.label_idx), dtype=torch.long),
            "valid_mask": torch.from_numpy(valid_mask),
            "valid_length": torch.tensor(valid_length, dtype=torch.long),
            "sample_id": str(row.sample_id),
        }


def make_loader(
    subset_manifest: pd.DataFrame,
    *,
    batch_size: int,
    shuffle: bool,
    seed: int,
):
    dataset = EventSNNDataset(data, subset_manifest)
    generator = torch.Generator().manual_seed(seed)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        generator=generator,
        worker_init_fn=worker_init_fn if NUM_WORKERS > 0 else None,
        pin_memory=torch.cuda.is_available(),
    )


In [ ]:
def make_split_loaders(master_seed: int):
    loaders = {}

    for name, split_name, shuffle in (
        ("train", "train", True),
        ("train_eval", "train", False),
        ("val", "val", False),
        ("test", "test", False),
    ):
        subset = FIXED_SPLIT_MANIFEST[
            FIXED_SPLIT_MANIFEST.split == split_name
        ]

        loaders[name] = make_loader(
            subset,
            batch_size=BATCH_SIZE,
            shuffle=shuffle,
            seed=derive_seed(master_seed, name, "loader"),
        )

    return loaders


## 6. Temporal aggregation helpers

In [ ]:
def relative_temporal_bin_counts(
    output_spikes: torch.Tensor,
    valid_lengths: torch.Tensor,
    n_bins: int,
) -> torch.Tensor:
    _, time_steps, _ = output_spikes.shape
    lengths = valid_lengths.to(output_spikes.device, dtype=torch.long)

    t = torch.arange(time_steps, device=output_spikes.device)[None, :]
    valid = t < lengths[:, None]

    bin_index = torch.div(
        t * n_bins,
        lengths[:, None],
        rounding_mode="floor",
    ).clamp_max(n_bins - 1)

    one_hot = F.one_hot(
        bin_index,
        num_classes=n_bins,
    ).to(output_spikes.dtype)

    one_hot = one_hot * valid.unsqueeze(-1).to(output_spikes.dtype)
    return torch.einsum("btn,btk->bnk", one_hot, output_spikes)


def fixed_duration_bin_counts(
    output_spikes: torch.Tensor,
    valid_mask: torch.Tensor,
) -> torch.Tensor:
    batch_size, time_steps, class_count = output_spikes.shape
    masked = output_spikes * valid_mask.unsqueeze(-1).to(output_spikes.dtype)

    target_steps = FIXED_BIN_COUNT * FIXED_BIN_STEPS
    if target_steps > time_steps:
        masked = F.pad(masked, (0, 0, 0, target_steps - time_steps))

    return masked.reshape(
        batch_size,
        FIXED_BIN_COUNT,
        FIXED_BIN_STEPS,
        class_count,
    ).sum(dim=2)


def last_fixed_window_counts(
    output_spikes: torch.Tensor,
    valid_lengths: torch.Tensor,
) -> torch.Tensor:
    _, time_steps, _ = output_spikes.shape
    lengths = valid_lengths.to(output_spikes.device, dtype=torch.long)

    t = torch.arange(time_steps, device=output_spikes.device)[None, :]
    start = (lengths - FIXED_BIN_STEPS).clamp_min(0)

    window_mask = (
        (t >= start[:, None])
        & (t < lengths[:, None])
    )

    return (
        output_spikes
        * window_mask.unsqueeze(-1).to(output_spikes.dtype)
    ).sum(dim=1)


## 7. Fixed no-bias SNN

In [ ]:
class SharedBackboneSNN(nn.Module):
    def __init__(self, num_classes: int, input_channels: int):
        super().__init__()

        self.num_classes = int(num_classes)
        self.input_channels = int(input_channels)

        spike_grad = surrogate.fast_sigmoid(slope=SURROGATE_SLOPE)

        self.fc1 = nn.Linear(input_channels, HIDDEN_WIDTH, bias=False)
        self.lif1 = snn.Synaptic(
            alpha=ALPHA,
            beta=BETA,
            threshold=THRESHOLD,
            spike_grad=spike_grad,
            learn_alpha=False,
            learn_beta=False,
            learn_threshold=False,
            reset_mechanism=RESET_MECHANISM,
        )

        self.fc2 = nn.Linear(HIDDEN_WIDTH, HIDDEN_WIDTH, bias=False)
        self.lif2 = snn.Synaptic(
            alpha=ALPHA,
            beta=BETA,
            threshold=THRESHOLD,
            spike_grad=spike_grad,
            learn_alpha=False,
            learn_beta=False,
            learn_threshold=False,
            reset_mechanism=RESET_MECHANISM,
        )

        self.fc_out = nn.Linear(HIDDEN_WIDTH, num_classes, bias=False)
        self.lif_out = snn.Synaptic(
            alpha=ALPHA,
            beta=BETA,
            threshold=THRESHOLD,
            spike_grad=spike_grad,
            learn_alpha=False,
            learn_beta=False,
            learn_threshold=False,
            reset_mechanism=RESET_MECHANISM,
        )

    @property
    def backbone_parameter_count(self):
        return sum(p.numel() for p in self.parameters())

    @property
    def spike_neuron_count(self):
        return 2 * HIDDEN_WIDTH + self.num_classes

    def forward(self, x, valid_mask):
        self.lif1.reset_mem()
        self.lif2.reset_mem()
        self.lif_out.reset_mem()

        spk1_rec, spk2_rec, spk_out_rec, mem_out_rec = [], [], [], []

        for step in range(x.shape[1]):
            spk1, _, _ = self.lif1(self.fc1(x[:, step, :]))
            spk2, _, _ = self.lif2(self.fc2(spk1))
            spk_out, _, mem_out = self.lif_out(self.fc_out(spk2))

            spk1_rec.append(spk1)
            spk2_rec.append(spk2)
            spk_out_rec.append(spk_out)
            mem_out_rec.append(mem_out)

        spk1_rec = torch.stack(spk1_rec, dim=1)
        spk2_rec = torch.stack(spk2_rec, dim=1)
        spk_out_rec = torch.stack(spk_out_rec, dim=1)
        mem_out_rec = torch.stack(mem_out_rec, dim=1)

        mask_f = valid_mask.unsqueeze(-1).to(spk_out_rec.dtype)
        total_valid_spikes = (
            (spk1_rec * mask_f).sum()
            + (spk2_rec * mask_f).sum()
            + (spk_out_rec * mask_f).sum()
        )

        return {
            "spk1": spk1_rec,
            "spk2": spk2_rec,
            "output_spikes": spk_out_rec,
            "output_membrane": mem_out_rec,
            "total_valid_spikes": total_valid_spikes,
        }


## 8. A–F objective implementations

In [ ]:
OBJECTIVE_NAMES = (
    "timestep_ce",
    "whole_count_ce",
    "relative_10bin_ce",
    "hybrid_relative10bin_count",
    "fixed_200ms_ce",
    "last_200ms_ce",
)


def masked_timestep_ce(output_spikes, labels, valid_mask):
    batch_size, time_steps, class_count = output_spikes.shape
    targets = labels[:, None].expand(batch_size, time_steps)

    loss_per_step = F.cross_entropy(
        output_spikes.reshape(batch_size * time_steps, class_count),
        targets.reshape(batch_size * time_steps),
        reduction="none",
    ).reshape(batch_size, time_steps)

    mask_f = valid_mask.to(loss_per_step.dtype)
    return (
        (loss_per_step * mask_f).sum()
        / mask_f.sum().clamp_min(1)
    )


class ObjectiveSNN(nn.Module):
    def __init__(self, objective, num_classes, input_channels, master_seed):
        super().__init__()

        if objective not in OBJECTIVE_NAMES:
            raise ValueError(f"Unknown objective {objective!r}")

        self.objective = objective
        self.num_classes = int(num_classes)

        seed_everything(derive_seed(master_seed, "shared_backbone_init"))
        self.backbone = SharedBackboneSNN(
            num_classes=num_classes,
            input_channels=input_channels,
        )

        self.readout_head = None

        if objective in {
            "relative_10bin_ce",
            "hybrid_relative10bin_count",
        }:
            seed_everything(derive_seed(master_seed, objective, "head_init"))
            self.readout_head = nn.Linear(
                N_RELATIVE_BINS * num_classes,
                num_classes,
                bias=True,
            )

        elif objective == "fixed_200ms_ce":
            seed_everything(derive_seed(master_seed, objective, "head_init"))
            self.readout_head = nn.Linear(
                FIXED_BIN_COUNT * num_classes,
                num_classes,
                bias=True,
            )

    @property
    def backbone_parameter_count(self):
        return self.backbone.backbone_parameter_count

    @property
    def readout_parameter_count(self):
        if self.readout_head is None:
            return 0
        return sum(p.numel() for p in self.readout_head.parameters())

    @property
    def total_parameter_count(self):
        return sum(p.numel() for p in self.parameters())

    def forward(self, x, valid_mask):
        return self.backbone(x, valid_mask)

    def full_counts(self, out, valid_mask):
        return (
            out["output_spikes"]
            * valid_mask.unsqueeze(-1).to(out["output_spikes"].dtype)
        ).sum(dim=1)

    def native_logits(self, out, valid_mask, valid_lengths):
        full_counts = self.full_counts(out, valid_mask)

        if self.objective in {"timestep_ce", "whole_count_ce"}:
            return full_counts

        if self.objective in {
            "relative_10bin_ce",
            "hybrid_relative10bin_count",
        }:
            bins = relative_temporal_bin_counts(
                out["output_spikes"],
                valid_lengths,
                N_RELATIVE_BINS,
            )
            return self.readout_head(bins.flatten(start_dim=1))

        if self.objective == "fixed_200ms_ce":
            bins = fixed_duration_bin_counts(
                out["output_spikes"],
                valid_mask,
            )
            return self.readout_head(bins.flatten(start_dim=1))

        if self.objective == "last_200ms_ce":
            return last_fixed_window_counts(
                out["output_spikes"],
                valid_lengths,
            )

        raise RuntimeError(self.objective)

    def training_loss(self, out, labels, valid_mask, valid_lengths):
        full_counts = self.full_counts(out, valid_mask)

        if self.objective == "timestep_ce":
            loss = masked_timestep_ce(
                out["output_spikes"],
                labels,
                valid_mask,
            )

        elif self.objective == "whole_count_ce":
            loss = F.cross_entropy(full_counts, labels)

        elif self.objective == "relative_10bin_ce":
            loss = F.cross_entropy(
                self.native_logits(out, valid_mask, valid_lengths),
                labels,
            )

        elif self.objective == "hybrid_relative10bin_count":
            loss = (
                F.cross_entropy(
                    self.native_logits(out, valid_mask, valid_lengths),
                    labels,
                )
                + HYBRID_COUNT_WEIGHT
                * F.cross_entropy(full_counts, labels)
            )

        elif self.objective == "fixed_200ms_ce":
            loss = F.cross_entropy(
                self.native_logits(out, valid_mask, valid_lengths),
                labels,
            )

        elif self.objective == "last_200ms_ce":
            loss = F.cross_entropy(
                self.native_logits(out, valid_mask, valid_lengths),
                labels,
            )

        else:
            raise RuntimeError(self.objective)

        if SPIKE_REGULARIZATION > 0:
            valid_neuron_time = (
                valid_mask.to(out["output_spikes"].dtype).sum().clamp_min(1)
                * self.backbone.spike_neuron_count
            )
            loss = (
                loss
                + SPIKE_REGULARIZATION
                * out["total_valid_spikes"]
                / valid_neuron_time
            )

        return loss


## 9. Metrics and zero-input diagnostics

In [ ]:
def classification_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(
            balanced_accuracy_score(y_true, y_pred)
        ),
        "macro_f1": float(
            f1_score(y_true, y_pred, average="macro", zero_division=0)
        ),
    }


@torch.no_grad()
def evaluate_objective_model(model, loader):
    model.eval()

    y_true, y_pred = [], []
    total_loss = 0.0
    sample_count = 0
    valid_steps = 0
    l1_spikes = l2_spikes = output_spikes = 0.0
    silent_output_samples = 0

    for batch in loader:
        x = batch["x"].to(DEVICE, non_blocking=True)
        y = batch["label"].to(DEVICE, non_blocking=True)
        mask = batch["valid_mask"].to(DEVICE, non_blocking=True)
        lengths = batch["valid_length"].to(DEVICE, non_blocking=True)

        out = model(x, mask)
        loss = model.training_loss(out, y, mask, lengths)
        logits = model.native_logits(out, mask, lengths)
        pred = logits.argmax(dim=1)

        batch_size = len(y)
        batch_valid_steps = int(mask.sum().item())

        total_loss += float(loss.item()) * batch_size
        sample_count += batch_size
        valid_steps += batch_valid_steps

        y_true.extend(y.cpu().tolist())
        y_pred.extend(pred.cpu().tolist())

        mask_f = mask.unsqueeze(-1).to(out["output_spikes"].dtype)
        batch_l1 = (out["spk1"] * mask_f).sum(dim=(1, 2))
        batch_l2 = (out["spk2"] * mask_f).sum(dim=(1, 2))
        batch_out = (out["output_spikes"] * mask_f).sum(dim=(1, 2))

        l1_spikes += float(batch_l1.sum().item())
        l2_spikes += float(batch_l2.sum().item())
        output_spikes += float(batch_out.sum().item())
        silent_output_samples += int((batch_out == 0).sum().item())

    metrics = classification_metrics(y_true, y_pred)
    metrics["loss"] = total_loss / max(sample_count, 1)
    metrics["l1_firing_rate"] = (
        l1_spikes / max(valid_steps * HIDDEN_WIDTH, 1)
    )
    metrics["l2_firing_rate"] = (
        l2_spikes / max(valid_steps * HIDDEN_WIDTH, 1)
    )
    metrics["output_firing_rate"] = (
        output_spikes / max(valid_steps * NUM_CLASSES, 1)
    )
    metrics["network_firing_rate"] = (
        (l1_spikes + l2_spikes + output_spikes)
        / max(valid_steps * (2 * HIDDEN_WIDTH + NUM_CLASSES), 1)
    )
    metrics["silent_output_fraction"] = (
        silent_output_samples / max(sample_count, 1)
    )
    metrics["samples"] = sample_count
    return metrics


@torch.no_grad()
def zero_input_activity(model, steps=ZERO_INPUT_DIAGNOSTIC_STEPS):
    model.eval()

    x = torch.zeros(
        1,
        steps,
        INPUT_CHANNELS,
        device=DEVICE,
    )
    mask = torch.ones(
        1,
        steps,
        dtype=torch.bool,
        device=DEVICE,
    )

    out = model(x, mask)

    return {
        "zero_l1_firing_rate": float(out["spk1"].mean().item()),
        "zero_l2_firing_rate": float(out["spk2"].mean().item()),
        "zero_output_firing_rate": float(
            out["output_spikes"].mean().item()
        ),
    }


## 10. Fixed-budget training; select best checkpoint by validation loss

In [ ]:
def train_one_objective(
    *,
    objective: str,
    master_seed: int,
    train_loader,
    train_eval_loader,
    val_loader,
):
    model = ObjectiveSNN(
        objective=objective,
        num_classes=NUM_CLASSES,
        input_channels=INPUT_CHANNELS,
        master_seed=master_seed,
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    best_state = None
    best_val_loss = np.inf
    best_epoch = -1
    firing_onset_epoch = None
    history_rows = []

    zero_before = zero_input_activity(model)

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()

        for batch in train_loader:
            x = batch["x"].to(DEVICE, non_blocking=True)
            y = batch["label"].to(DEVICE, non_blocking=True)
            mask = batch["valid_mask"].to(DEVICE, non_blocking=True)
            lengths = batch["valid_length"].to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            out = model(x, mask)
            loss = model.training_loss(out, y, mask, lengths)

            if not torch.isfinite(loss):
                raise FloatingPointError(
                    f"Non-finite loss for objective={objective}, seed={master_seed}"
                )

            loss.backward()

            if GRAD_CLIP_NORM is not None:
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    GRAD_CLIP_NORM,
                )

            optimizer.step()

        train_metrics = evaluate_objective_model(
            model,
            train_eval_loader,
        )
        val_metrics = evaluate_objective_model(
            model,
            val_loader,
        )

        if (
            firing_onset_epoch is None
            and val_metrics["output_firing_rate"] > FIRING_ONSET_THRESHOLD
        ):
            firing_onset_epoch = epoch

        if val_metrics["loss"] < best_val_loss - 1e-12:
            best_val_loss = val_metrics["loss"]
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())

        history_rows.append(
            {
                "seed": master_seed,
                "objective": objective,
                "epoch": epoch,
                "train_loss": train_metrics["loss"],
                "train_accuracy": train_metrics["accuracy"],
                "train_balanced_accuracy": train_metrics["balanced_accuracy"],
                "train_macro_f1": train_metrics["macro_f1"],
                "train_l1_firing_rate": train_metrics["l1_firing_rate"],
                "train_l2_firing_rate": train_metrics["l2_firing_rate"],
                "train_output_firing_rate": train_metrics["output_firing_rate"],
                "train_silent_output_fraction": train_metrics["silent_output_fraction"],
                "val_loss": val_metrics["loss"],
                "val_accuracy": val_metrics["accuracy"],
                "val_balanced_accuracy": val_metrics["balanced_accuracy"],
                "val_macro_f1": val_metrics["macro_f1"],
                "val_l1_firing_rate": val_metrics["l1_firing_rate"],
                "val_l2_firing_rate": val_metrics["l2_firing_rate"],
                "val_output_firing_rate": val_metrics["output_firing_rate"],
                "val_silent_output_fraction": val_metrics["silent_output_fraction"],
            }
        )

        if epoch == 1 or epoch % 10 == 0 or epoch == NUM_EPOCHS:
            print(
                f"epoch {epoch:>3d} | "
                f"train loss={train_metrics['loss']:.4f} "
                f"train BA={train_metrics['balanced_accuracy']:.4f} | "
                f"val loss={val_metrics['loss']:.4f} "
                f"val BA={val_metrics['balanced_accuracy']:.4f} | "
                f"out FR={val_metrics['output_firing_rate']:.4f}"
            )

    if best_state is None:
        raise RuntimeError("No best checkpoint selected")

    model.load_state_dict(best_state)
    zero_after = zero_input_activity(model)

    return {
        "model": model,
        "history": pd.DataFrame(history_rows),
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "firing_onset_epoch": firing_onset_epoch,
        "zero_before": zero_before,
        "zero_after": zero_after,
    }


## 11. Common FullCount diagnostic

In [ ]:
@torch.no_grad()
def evaluate_common_full_count(model, loader):
    """Use the same no-extra-parameter FullCount rule for every trained backbone."""
    model.eval()

    y_true = []
    y_pred = []

    for batch in loader:
        x = batch["x"].to(DEVICE, non_blocking=True)
        y = batch["label"].to(DEVICE, non_blocking=True)
        mask = batch["valid_mask"].to(DEVICE, non_blocking=True)

        out = model(x, mask)

        counts = (
            out["output_spikes"]
            * mask.unsqueeze(-1).to(out["output_spikes"].dtype)
        ).sum(dim=1)

        pred = counts.argmax(dim=1)

        y_true.extend(y.cpu().tolist())
        y_pred.extend(pred.cpu().tolist())

    return classification_metrics(y_true, y_pred)


## 12. Main A–F × five-seed sweep

In [ ]:
for objective in OBJECTIVES_TO_RUN:
    if objective not in OBJECTIVE_NAMES:
        raise ValueError(
            f"Unknown objective {objective!r}; expected one of {OBJECTIVE_NAMES}"
        )

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

result_rows = []
history_frames = []

for master_seed in SEEDS:
    print("=" * 100)
    print("MASTER SEED:", master_seed)

    for objective in OBJECTIVES_TO_RUN:
        print("-" * 100)
        print("OBJECTIVE:", objective)

        # Rebuilt with the same seeds -> paired data order across objectives.
        loaders = make_split_loaders(master_seed)

        run = train_one_objective(
            objective=objective,
            master_seed=master_seed,
            train_loader=loaders["train"],
            train_eval_loader=loaders["train_eval"],
            val_loader=loaders["val"],
        )

        model = run["model"]
        history_frames.append(run["history"])

        split_metrics = {
            "train": evaluate_objective_model(model, loaders["train_eval"]),
            "val": evaluate_objective_model(model, loaders["val"]),
            "test": evaluate_objective_model(model, loaders["test"]),
        }

        common_fullcount = {
            "train": evaluate_common_full_count(model, loaders["train_eval"]),
            "val": evaluate_common_full_count(model, loaders["val"]),
            "test": evaluate_common_full_count(model, loaders["test"]),
        }

        row = {
            "experiment": EXPERIMENT_ID,
            "seed": master_seed,
            "objective": objective,
            "best_epoch_by_val_loss": run["best_epoch"],
            "best_val_loss": run["best_val_loss"],
            "firing_onset_epoch": (
                np.nan
                if run["firing_onset_epoch"] is None
                else run["firing_onset_epoch"]
            ),
            "input_channels": INPUT_CHANNELS,
            "hidden_width": HIDDEN_WIDTH,
            "num_classes": NUM_CLASSES,
            "backbone_params": model.backbone_parameter_count,
            "readout_params": model.readout_parameter_count,
            "total_params": model.total_parameter_count,
            **run["zero_after"],
        }

        for split_name, metrics in split_metrics.items():
            for metric_name, value in metrics.items():
                row[f"{split_name}_{metric_name}"] = value

        for split_name, metrics in common_fullcount.items():
            for metric_name, value in metrics.items():
                row[
                    f"{split_name}_common_fullcount_{metric_name}"
                ] = value

        result_rows.append(row)

        print(
            f"best epoch={run['best_epoch']} | "
            f"native train BA={split_metrics['train']['balanced_accuracy']:.4f} | "
            f"val BA={split_metrics['val']['balanced_accuracy']:.4f} | "
            f"test BA={split_metrics['test']['balanced_accuracy']:.4f} | "
            f"common FullCount test BA="
            f"{common_fullcount['test']['balanced_accuracy']:.4f}"
        )
        print("zero-input after best checkpoint:", run["zero_after"])

        del model

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

results = pd.DataFrame(result_rows)
histories = pd.concat(history_frames, ignore_index=True)

results.to_csv(
    RESULTS_DIR / "experiment_2_1_objective_results.csv",
    index=False,
)
histories.to_csv(
    RESULTS_DIR / "experiment_2_1_objective_histories.csv",
    index=False,
)

display(results)


## 13. Aggregate summary

In [ ]:
summary = (
    results.groupby("objective", as_index=False)
    .agg(
        mean_train_ba=("train_balanced_accuracy", "mean"),
        sd_train_ba=("train_balanced_accuracy", "std"),
        mean_val_ba=("val_balanced_accuracy", "mean"),
        sd_val_ba=("val_balanced_accuracy", "std"),
        mean_test_ba=("test_balanced_accuracy", "mean"),
        sd_test_ba=("test_balanced_accuracy", "std"),
        mean_test_macro_f1=("test_macro_f1", "mean"),
        mean_common_fullcount_test_ba=(
            "test_common_fullcount_balanced_accuracy",
            "mean",
        ),
        mean_output_firing_rate=("test_output_firing_rate", "mean"),
        mean_silent_output_fraction=("test_silent_output_fraction", "mean"),
        mean_firing_onset_epoch=("firing_onset_epoch", "mean"),
        backbone_params=("backbone_params", "first"),
        readout_params=("readout_params", "first"),
        total_params=("total_params", "first"),
    )
)

order_map = {
    name: i
    for i, name in enumerate(OBJECTIVES_TO_RUN)
}

summary["_order"] = summary["objective"].map(order_map)
summary = (
    summary.sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)

summary.to_csv(
    RESULTS_DIR / "experiment_2_1_objective_summary.csv",
    index=False,
)

display(summary)


## 14. Native objective performance

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(OBJECTIVES_TO_RUN), dtype=float)
offsets = {"train": -0.22, "val": 0.0, "test": 0.22}

for split_name in ("train", "val", "test"):
    means = []
    sds = []

    for objective in OBJECTIVES_TO_RUN:
        sdf = results[results.objective == objective]
        means.append(
            sdf[f"{split_name}_balanced_accuracy"].mean()
        )
        sds.append(
            sdf[f"{split_name}_balanced_accuracy"].std(ddof=1)
        )

    ax.errorbar(
        x + offsets[split_name],
        means,
        yerr=sds,
        marker="o",
        linestyle="none",
        capsize=4,
        label=split_name,
    )

ax.axhline(1.0 / NUM_CLASSES, linestyle="--", label="chance BA")
ax.set_xticks(x, OBJECTIVES_TO_RUN, rotation=25, ha="right")
ax.set_ylabel("Balanced accuracy")
ax.set_xlabel("Training objective")
ax.set_title(
    "Experiment 2.1 — Training-objective ablation\n"
    "Native objective readout, mean ± SD across paired seeds"
)
ax.grid(True, axis="y", alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()


## 15. Firing-onset comparison

In [ ]:
display(
    summary[
        [
            "objective",
            "mean_firing_onset_epoch",
            "mean_output_firing_rate",
            "mean_silent_output_fraction",
        ]
    ]
)

fig, ax = plt.subplots(figsize=(12, 6))

for objective in OBJECTIVES_TO_RUN:
    sdf = histories[histories.objective == objective]
    mean_curve = sdf.groupby("epoch")["val_output_firing_rate"].mean()

    ax.plot(
        mean_curve.index,
        mean_curve.values,
        label=objective,
    )

ax.axhline(
    FIRING_ONSET_THRESHOLD,
    linestyle="--",
    label="firing-onset threshold",
)
ax.set_xlabel("Epoch")
ax.set_ylabel("Mean validation output firing rate")
ax.set_title("How quickly each objective escapes the silent regime")
ax.grid(True, alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()


## 16. Validation-BA trajectory

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for objective in OBJECTIVES_TO_RUN:
    sdf = histories[histories.objective == objective]
    mean_curve = sdf.groupby("epoch")["val_balanced_accuracy"].mean()

    ax.plot(
        mean_curve.index,
        mean_curve.values,
        label=objective,
    )

ax.axhline(1.0 / NUM_CLASSES, linestyle="--", label="chance BA")
ax.set_xlabel("Epoch")
ax.set_ylabel("Mean validation balanced accuracy")
ax.set_title("Validation BA trajectory by training objective")
ax.grid(True, alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()


## 17. Save provenance

In [ ]:
provenance = {
    "experiment_id": EXPERIMENT_ID,
    "dataset_roots": [str(Path(p).resolve()) for p in DATASET_ROOTS],
    "seeds": list(SEEDS),
    "split_seed": SPLIT_SEED,
    "included_labels": list(INCLUDED_LABELS),
    "class_to_idx": CLASS_TO_IDX,
    "sampling_rate_hz": SAMPLING_RATE_HZ,
    "event_representation": EVENT_REPRESENTATION,
    "event_feature_schema": EVENT_FEATURE_SCHEMA,
    "encoder_spec_sha256": ENCODER_SPEC_SHA256,
    "input_channels": INPUT_CHANNELS,
    "hidden_width": HIDDEN_WIDTH,
    "num_classes": NUM_CLASSES,
    "objectives": list(OBJECTIVES_TO_RUN),
    "objective_definitions": {
        "timestep_ce": "masked per-valid-timestep CE on output spikes",
        "whole_count_ce": "CE on whole-valid-sequence output spike counts",
        "relative_10bin_ce": "relative 10-bin counts -> Linear -> CE",
        "hybrid_relative10bin_count": (
            "relative 10-bin CE + "
            f"{HYBRID_COUNT_WEIGHT} * whole-count CE"
        ),
        "fixed_200ms_ce": (
            f"fixed {FIXED_BIN_MS} ms count bins -> Linear -> CE"
        ),
        "last_200ms_ce": (
            f"CE on output counts from final valid {FIXED_BIN_MS} ms only"
        ),
    },
    "fixed_bin_ms": FIXED_BIN_MS,
    "fixed_bin_steps": FIXED_BIN_STEPS,
    "fixed_bin_count": FIXED_BIN_COUNT,
    "relative_bin_count": N_RELATIVE_BINS,
    "tau_syn_ms": TAU_SYN_MS,
    "tau_mem_ms": TAU_MEM_MS,
    "alpha": ALPHA,
    "beta": BETA,
    "threshold": THRESHOLD,
    "surrogate_slope": SURROGATE_SLOPE,
    "reset_mechanism": RESET_MECHANISM,
    "snn_linear_bias": False,
    "learn_alpha": False,
    "learn_beta": False,
    "learn_threshold": False,
    "num_epochs": NUM_EPOCHS,
    "early_stopping": False,
    "checkpoint_selection": (
        "minimum validation loss after full fixed epoch budget"
    ),
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "spike_regularization": SPIKE_REGULARIZATION,
    "firing_onset_threshold": FIRING_ONSET_THRESHOLD,
}

with open(
    RESULTS_DIR / "experiment_2_1_objective_provenance.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(provenance, f, indent=2, ensure_ascii=False)

print("Saved to:", RESULTS_DIR)


## Decision guide

A useful objective should:

1. push **Train BA** clearly above chance across seeds;
2. improve validation BA, not just one lucky initialization;
3. escape the silent regime reliably;
4. avoid saturated output firing;
5. keep zero-input firing at 0.

Key comparisons:

- **A vs B:** does whole-gesture supervision fix the mismatch between early timesteps and final letter identity?
- **C vs E:** is gesture-relative phase necessary, or can causal fixed 200 ms chunks retain most of the benefit?
- **E vs F:** is the final 200 ms alone enough, or must multiple local patterns across the gesture be integrated?
- **C vs D:** does a small whole-count term stabilize temporal training and make total output activity more class-selective?

`common FullCount test BA` is reported for every trained backbone as a secondary diagnostic. It shows whether an objective also shapes a simple class-selective total-spike representation.

Do not start width/depth sweeps until at least one objective is clearly trainable.
